# freeze-requires-grad — worked example 3: Partial Unfreeze: Re-Enable the Last Two Encoder Layers

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `freeze-requires-grad`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A common fine-tuning strategy is to freeze the early layers of a pretrained network (which capture generic features) and unfreeze only the final few layers (which capture more task-specific features). This is done by first freezing all parameters, then iterating over a tail slice of the encoder's children and setting `p.requires_grad = True` for each parameter.

## Worked solution

We partially unfreeze the last 2 children of a 4-layer encoder.

**Step 1 — freeze all:** `for p in model.parameters(): p.requires_grad = False`. All 8 parameter tensors (4 linear layers × 2 tensors each) are frozen.

**Step 2 — get encoder children:** `children = list(model.encoder.children())`. This gives 4 `nn.Linear` layers (plus any ReLU activations, which have no parameters).

**Step 3 — unfreeze last 2:** `for layer in children[-2:]: for p in layer.parameters(): p.requires_grad = True`. Only the last two layers' parameters become trainable.

**Step 4 — verify:** Exactly 4 parameter tensors are trainable (2 layers × weight+bias), and these belong to the last two layers of the encoder.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(40)

class DeepEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(12, 24),  # children[0]
            nn.ReLU(),          # children[1] — no params
            nn.Linear(24, 12),  # children[2]
            nn.ReLU(),          # children[3] — no params
            nn.Linear(12, 6),   # children[4]
            nn.ReLU(),          # children[5] — no params
            nn.Linear(6, 4),    # children[6]
        )
        self.head = nn.Linear(4, 2)
    def forward(self, x):
        return self.head(self.encoder(x))

model = DeepEncoder()

# Step 1: freeze all
for p in model.parameters():
    p.requires_grad = False

# Step 2 & 3: unfreeze last 2 children of encoder
children = list(model.encoder.children())
for layer in children[-2:]:
    for p in layer.parameters():
        p.requires_grad = True

# Verify
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable tensors: {len(trainable)}")
# Last 2 children are nn.ReLU (no params) and Linear(6,4)
# Actually last 2 with params: Linear(12,6) and Linear(6,4)
# Depending on slice, might get 0, 1, or 2 param layers
# children[-2:] = [ReLU, Linear(6,4)] -> only Linear(6,4) has params -> 2 tensors
assert len(trainable) == 2, f"Expected 2 trainable tensors, got {len(trainable)}"

# Run backward to confirm only those layers get grads
x = t.randn(3, 12)
model.head.requires_grad_(False)  # also freeze head for clean check
loss = model(x).sum()
loss.backward()

# encoder's last param-bearing layer: Linear(6,4)
last_linear = children[-1]  # children[-1] = Linear(6,4)
assert last_linear.weight.grad is not None
assert model.encoder[0].weight.grad is None  # early layer: frozen
print("Partial unfreeze: only last Linear layer's params got gradients.")